# BlinkMimic-T4 正式训练（双卡 DDP + fp16 + 自动备份）

直接 Run All。每次运行新建 `results/blinkmimic_t4_ddp/run_<时间戳>/`，不会覆盖旧日志和 checkpoint。

## 档位

| 档位 | envs/卡 | pi_batch | wm_batch | policy 宽度 | compile |
|---|---|---|---|---|---|
| baseline | 2048 | 1024 | 1024 | 1024-512-256 | 关 |
| high_util | 4096 | 2048 | 2048 | 2048-1024-512-256 | 开 |
| max（当前默认） | 8192 | 3072 | 2048 | 2048-1024-512-256 | 开 + compile_more |

`max` 的目标是把 8 核和 15 GB 显存尽量用满。CPU 线程按 `CPU_COUNT / 2` 自动分配，8 核时每 rank 4 线程。

## 数据与精度

- 40 段 LAFAN1 G1 重定向 CSV，30 fps，由仓库内部重采样到 50 Hz
- fp16 + 静态 loss scale 256
- 3500 iterations，horizon 32
- checkpoint：每 500 iteration 或 10 分钟，完整状态
- 后台线程上传 checkpoint 和日志到 Hugging Face 私有仓库

## 备份

- HF token 从 Kaggle Secrets 读取（`HF_TOKEN`），不写进 Notebook
- 自动创建私有仓库 `<hf_user>/proj2-blinkmimic-t4`
- 断点续训：`RESUME_RUN_ID` 设为旧 run 时间戳，会从 HF 拉 `last.pt`

## 出问题时的降档顺序

1. OOM：`PER_RANK_PI_BATCH` 3072 → 2048
2. 仍 OOM：`PER_RANK_ENVS` 8192 → 4096
3. Triton/sm_75 编译报错：`DISABLE_COMPILE = True`
4. NaN：`--fp16_loss_scale` 256 → 64

仅仿真验证。所有数字来自本次 Kaggle 日志。LAFAN1 数据只放私有 Dataset，不提交 GitHub，不上传公开仓库。


In [ ]:
from pathlib import Path
import os
import time

REPO_DIR = Path('/kaggle/working/proj2-humanoid-mimic/third_party/blinkmimic')
REPO_URL = 'https://github.com/fuyichen0725/blinkmimic.git'
COMMIT = 'd264658d6c9ce61fb52f9f0f8985a369f3ccc409'
MOTION_SRC = Path('/kaggle/input/datasets/machineabao/smoke1')
BASE_RESULTS = Path('/kaggle/working/proj2-humanoid-mimic/results/blinkmimic_t4_ddp')

DATA_MODE = 'full40'
RESUME_RUN_ID = ''
RUN_ID = RESUME_RUN_ID or time.strftime('%Y%m%d_%H%M%S')
LOG_ROOT = BASE_RESULTS / f'run_{RUN_ID}'
HF_REPO_NAME = 'proj2-blinkmimic-t4'

PROFILE = 'max'  # 'baseline' | 'high_util' | 'max'
if PROFILE == 'baseline':
    PER_RANK_ENVS = 2048
    PER_RANK_PI_BATCH = 1024
    PER_RANK_WM_BATCH = 1024
    POLICY_HIDDEN = '1024,512,256'
    DISABLE_COMPILE = True
    COMPILE_MORE = False
elif PROFILE == 'high_util':
    PER_RANK_ENVS = 4096
    PER_RANK_PI_BATCH = 2048
    PER_RANK_WM_BATCH = 2048
    POLICY_HIDDEN = '2048,1024,512,256'
    DISABLE_COMPILE = False
    COMPILE_MORE = True
elif PROFILE == 'max':
    PER_RANK_ENVS = 8192
    PER_RANK_PI_BATCH = 3072
    PER_RANK_WM_BATCH = 2048
    POLICY_HIDDEN = '2048,1024,512,256'
    DISABLE_COMPILE = False
    COMPILE_MORE = True
else:
    raise ValueError(f'未知 PROFILE: {PROFILE}')

HF_TOKEN = os.environ.get('HF_TOKEN', '')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or HF_TOKEN
except Exception as exc:
    print(f'kaggle_secrets unavailable: {exc}')
print('HF_TOKEN=' + ('set' if HF_TOKEN else 'MISSING'))

WORLD_SIZE = 2
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 2)
CPU_THREADS_PER_RANK = max(1, CPU_COUNT // WORLD_SIZE)
ITERATIONS = 3500
TIMEOUT_S = 10 * 3600
FP16_LOSS_SCALE = '256'

T4_FLAGS = [
    '--motion_dir', 'data/lafan1_g1',
    '--num_envs', str(PER_RANK_ENVS),
    '--iterations', str(ITERATIONS),
    '--pi_batch', str(PER_RANK_PI_BATCH),
    '--wm_batch', str(PER_RANK_WM_BATCH),
    '--wm_batches', '5',
    '--horizon', '32',
    '--wm_hidden', '1024,1024,1024',
    '--enc_hidden', POLICY_HIDDEN,
    '--dec_hidden', POLICY_HIDDEN,
    '--amp', 'fp16',
    '--fp16_loss_scale', FP16_LOSS_SCALE,
    '--lr', '4e-4',
    '--pi_lr_mult', '4',
    '--lr_decay_ret', '0.75',
    '--lr_decay_len', '300',
    '--lr_decay_by', '1400',
    '--pi_ema', '0.995',
    '--no_sync',
    '--fused_adam',
    '--fused_rot',
    '--fail_reset', '0.3',
    '--contact_reward', '0.25',
    '--wm_structured',
    '--wm_mirror', '0.5',
    '--pi_opt', 'muon',
    '--wm_opt', 'muon',
    '--pi_ref_skip',
    '--wm_only_iters', '0',
    '--bin_fix',
    '--ref_contact_fix',
    '--ckpt_minutes', '10',
    '--ckpt_every', '500',
    '--log_every', '50',
    '--max_hours', '8',
]
if COMPILE_MORE:
    T4_FLAGS.append('--compile_more')

LOG_ROOT.mkdir(parents=True, exist_ok=True)
print(f'RUN_ID={RUN_ID}')
print(f'LOG_ROOT={LOG_ROOT}')
print(f'PROFILE={PROFILE} data={DATA_MODE}')
print(f'envs/rank={PER_RANK_ENVS} pi_batch={PER_RANK_PI_BATCH} wm_batch={PER_RANK_WM_BATCH}')
print(f'policy_hidden={POLICY_HIDDEN} compile={not DISABLE_COMPILE} compile_more={COMPILE_MORE}')
print(f'cpu_count={CPU_COUNT} threads_per_rank={CPU_THREADS_PER_RANK}')


In [ ]:
import subprocess

info = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv,noheader'],
    text=True,
)
print(info.strip())
gpu_count = len([line for line in info.strip().splitlines() if line.strip()])
print(f'gpu_count={gpu_count}')
if gpu_count != WORLD_SIZE:
    raise RuntimeError(f'需要 {WORLD_SIZE} 张 GPU，当前 {gpu_count} 张')


In [ ]:
import shutil
import subprocess

def run(command, cwd=None, check=True):
    print('$', ' '.join(str(item) for item in command))
    return subprocess.run(command, cwd=cwd, check=check, text=True)

REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
lock = REPO_DIR / '.git' / 'index.lock'
if lock.exists():
    lock.unlink()
if (REPO_DIR / '.git').exists():
    probe = subprocess.run(['git', 'rev-parse', '--git-dir'], cwd=REPO_DIR, capture_output=True, text=True)
    if probe.returncode != 0:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)])
run(['git', 'fetch', '--depth', '1', 'origin', COMMIT], cwd=REPO_DIR)
run(['git', 'checkout', '--detach', COMMIT], cwd=REPO_DIR)
run(['git', 'reset', '--hard', COMMIT], cwd=REPO_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print(f'git_head={head}')
if head != COMMIT:
    raise RuntimeError('commit 校验失败')


In [ ]:
import base64
import subprocess

PATCH_B64 = ''.join([
    '...',
    'ZWQuZGVzdHJveV9wcm9jZXNzX2dyb3VwKCkK',
])

MARKER = 'NCCL cannot broadcast CPU buffers'
train_py = REPO_DIR / 'src' / 'blinkmimic' / 'train.py'
if MARKER in train_py.read_text():
    print('patch_already_applied')
else:
    subprocess.run(['git', 'reset', '--hard', COMMIT], cwd=REPO_DIR, check=True)
    patch_path = REPO_DIR / 't4_ddp_fp16.patch'
    patch_path.write_bytes(base64.b64decode(PATCH_B64))
    subprocess.run(['git', 'apply', '--check', str(patch_path)], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'apply', str(patch_path)], cwd=REPO_DIR, check=True)
    print('patch_applied')
    diff = subprocess.check_output(['git', 'diff'], cwd=REPO_DIR, text=True)
    (LOG_ROOT / 'applied.patch').write_text(diff)
    print(f'diff_lines={len(diff.splitlines())}')

assert MARKER in train_py.read_text()
assert 'fp16_loss_scale' in train_py.read_text()
print('patch_verify=PASS')


In [ ]:
import subprocess

def run(command, cwd=None, check=True):
    print('$', ' '.join(str(item) for item in command))
    return subprocess.run(command, cwd=cwd, check=check, text=True)

run(['uv', 'sync', '--frozen'], cwd=REPO_DIR)
check = '''
import importlib.metadata as m, sys, torch
print(f"python={sys.version.split()[0]}")
print(f"torch={torch.__version__} cuda={torch.version.cuda}")
print(f"bf16_supported={torch.cuda.is_bf16_supported()}")
for n in ("mjlab", "mujoco", "warp-lang"):
    try:
        print(f"{n}={m.version(n)}")
    except m.PackageNotFoundError:
        print(f"{n}=NOT_FOUND")
'''
run(['uv', 'run', 'python', '-c', check], cwd=REPO_DIR)


In [ ]:
import json
import subprocess
import urllib.request
from pathlib import Path

motion_dir = REPO_DIR / 'data' / 'lafan1_g1'
motion_dir.mkdir(parents=True, exist_ok=True)

if DATA_MODE == 'full40':
    for item in motion_dir.glob('*.csv'):
        item.unlink()
    subprocess.run(['bash', 'tools/download_data.sh'], cwd=REPO_DIR, check=True)
elif DATA_MODE == 'smoke2':
    if not MOTION_SRC.exists():
        raise FileNotFoundError(f'动作目录不存在: {MOTION_SRC}')
    src_csv = sorted(MOTION_SRC.glob('*.csv'))
    if not src_csv:
        raise FileNotFoundError(f'{MOTION_SRC} 下没有 CSV')
    for source in src_csv:
        target = motion_dir / source.name
        if target.exists() or target.is_symlink():
            target.unlink()
        target.symlink_to(source)
else:
    raise ValueError(f'未知 DATA_MODE: {DATA_MODE}')

csv_files = sorted(motion_dir.glob('*.csv'))
print(f'csv_count={len(csv_files)}')
for item in csv_files[:5]:
    print(' ', item.name, item.stat().st_size // 1024, 'KB')
if len(csv_files) < 8:
    raise RuntimeError(f'动作数不足 8: {len(csv_files)}')

desc = REPO_DIR / 'data' / 'g1_description'
mjcf = desc / 'g1_29dof.xml'
if not mjcf.exists():
    HF = 'https://huggingface.co/datasets/lvhaidong/LAFAN1_Retargeting_Dataset'
    API = 'https://huggingface.co/api/datasets/lvhaidong/LAFAN1_Retargeting_Dataset/tree/main'
    meshes = desc / 'meshes'
    meshes.mkdir(parents=True, exist_ok=True)

    def download(url, out):
        if out.exists() and out.stat().st_size > 0:
            return
        urllib.request.urlretrieve(url, out)

    download(f'{HF}/resolve/main/robot_description/g1/g1_29dof_rev_1_0.urdf', desc / 'g1_29dof_rev_1_0.urdf')
    with urllib.request.urlopen(f'{API}/robot_description/g1/meshes') as response:
        entries = json.load(response)
    for entry in entries:
        download(f'{HF}/resolve/main/{entry["path"]}', meshes / Path(entry['path']).name)
    subprocess.run(['uv', 'run', 'python', 'tools/urdf_to_mjcf.py'], cwd=REPO_DIR, check=True)
print(f'mjcf_exists={mjcf.exists()} size={mjcf.stat().st_size if mjcf.exists() else 0}')


In [ ]:
import subprocess

command = ['uv', 'run', 'python', 'tools/test_env.py', '--num_envs', '64', '--motion_dir', 'data/lafan1_g1']
print('$', ' '.join(command))
result = subprocess.run(command, cwd=REPO_DIR, text=True)
if result.returncode != 0:
    raise RuntimeError(f'test_env 失败 exit={result.returncode}')
print('test_env=PASS')


In [ ]:
import csv
import os
import subprocess
import sys
import threading
import time

try:
    import huggingface_hub  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)

HF_REPO = None
BACKUP_ENABLED = bool(HF_TOKEN)
if BACKUP_ENABLED:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    hf_user = api.whoami()['name']
    HF_REPO = f'{hf_user}/{HF_REPO_NAME}'
    api.create_repo(repo_id=HF_REPO, repo_type='model', private=True, exist_ok=True)
    print(f'HF_REPO={HF_REPO}')
else:
    print('HF backup disabled: 请把 HF_TOKEN 加入 Kaggle Secrets')

resume_args = []
if RESUME_RUN_ID:
    if not BACKUP_ENABLED:
        raise RuntimeError('RESUME_RUN_ID 需要 HF_TOKEN')
    from huggingface_hub import hf_hub_download
    resumed = hf_hub_download(
        repo_id=HF_REPO,
        filename=f'runs/run_{RUN_ID}/last.pt',
        local_dir=str(LOG_ROOT),
        token=HF_TOKEN,
    )
    resume_args = ['--resume', resumed]
    print(f'resume_from={resumed}')

uploaded = set()
last_log_upload = [0.0]

def hf_backup_once(final=False):
    if not BACKUP_ENABLED:
        return
    from huggingface_hub import HfApi
    client = HfApi(token=HF_TOKEN)
    now = time.time()
    upload_logs = final or (now - last_log_upload[0] > 300)
    for path in sorted(LOG_ROOT.rglob('*')):
        if not path.is_file() or 'runs' in path.parts:
            continue
        if path.suffix not in {'.pt', '.csv', '.log', '.json'}:
            continue
        if path.suffix != '.pt' and not upload_logs:
            continue
        stat = path.stat()
        key = (str(path.relative_to(LOG_ROOT)), stat.st_mtime_ns, stat.st_size)
        if key in uploaded:
            continue
        client.upload_file(
            path_or_fileobj=str(path),
            path_in_repo=f'runs/run_{RUN_ID}/{path.relative_to(LOG_ROOT)}',
            repo_id=HF_REPO,
            repo_type='model',
        )
        uploaded.add(key)
        print(f'[hf] uploaded {path.relative_to(LOG_ROOT)} ({stat.st_size / 1e6:.1f} MB)')
    if upload_logs:
        last_log_upload[0] = now

stop_backup = threading.Event()

def backup_loop():
    while not stop_backup.wait(60):
        try:
            hf_backup_once()
        except Exception as exc:
            print(f'[hf] warning: {exc}')

backup_thread = threading.Thread(target=backup_loop, daemon=True)
backup_thread.start()

gpu_log = LOG_ROOT / 'gpu_samples.csv'
run_log = LOG_ROOT / 'torchrun.log'
stop_monitor = threading.Event()

def monitor_gpu():
    with open(gpu_log, 'w', newline='') as output:
        writer = csv.writer(output)
        writer.writerow(['timestamp', 'gpu_index', 'memory_used_mib', 'utilization_pct'])
        while not stop_monitor.is_set():
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=timestamp,index,memory.used,utilization.gpu', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, check=False,
            )
            for line in result.stdout.splitlines():
                writer.writerow([part.strip() for part in line.split(',')])
            output.flush()
            stop_monitor.wait(2)

monitor = threading.Thread(target=monitor_gpu, daemon=True)
monitor.start()

env = os.environ.copy()
env['OMP_NUM_THREADS'] = str(CPU_THREADS_PER_RANK)
env['MKL_NUM_THREADS'] = str(CPU_THREADS_PER_RANK)
env['OPENBLAS_NUM_THREADS'] = str(CPU_THREADS_PER_RANK)
env['NUMEXPR_NUM_THREADS'] = str(CPU_THREADS_PER_RANK)
env['NCCL_DEBUG'] = 'WARN'
env['NCCL_P2P_DISABLE'] = '1'
env['MUJOCO_GL'] = 'egl'
env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
if DISABLE_COMPILE:
    env['TORCH_COMPILE_DISABLE'] = '1'

command = [
    'uv', 'run', 'torchrun',
    '--standalone',
    '--nproc_per_node', str(WORLD_SIZE),
    '-m', 'blinkmimic.train',
    '--log_dir', str(LOG_ROOT),
    *T4_FLAGS,
    *resume_args,
]
print('$', ' '.join(command))
started = time.time()
handle = open(run_log, 'w')
process = subprocess.Popen(
    command, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

def pump():
    for line in process.stdout:
        print(line, end='')
        handle.write(line)
        handle.flush()

reader = threading.Thread(target=pump, daemon=True)
reader.start()
try:
    returncode = process.wait(timeout=TIMEOUT_S)
except subprocess.TimeoutExpired:
    process.terminate()
    try:
        returncode = process.wait(timeout=60)
    except subprocess.TimeoutExpired:
        process.kill()
        returncode = process.wait()
    print(f'timeout_after_seconds={TIMEOUT_S}')
reader.join(timeout=10)
handle.close()
stop_monitor.set()
monitor.join(timeout=5)
stop_backup.set()
try:
    hf_backup_once(final=True)
except Exception as exc:
    print(f'[hf] final warning: {exc}')
backup_thread.join(timeout=30)

print(f'train_wall_seconds={time.time() - started:.1f}')
print(f'exit_code={returncode}')

rows = list(csv.DictReader(open(gpu_log)))
for gpu_index in sorted({row['gpu_index'] for row in rows}):
    values = [int(row['memory_used_mib']) for row in rows if row['gpu_index'] == gpu_index]
    print(f'gpu_{gpu_index}_peak_memory_mib={max(values) if values else 0}')

log_path = LOG_ROOT / 'train_log.csv'
if log_path.exists():
    lines = log_path.read_text(errors='replace').splitlines()
    print(f'log_rows={len(lines) - 1}')
    for line in lines[-8:]:
        print(line)
    if any('nan' in line.lower() for line in lines[1:]):
        print('warning=NaN_IN_LOG')
else:
    print('train_log_missing; torchrun tail:')
    lines = run_log.read_text(errors='replace').splitlines()
    for line in lines[-60:]:
        print(line)

print(f'LOG_ROOT={LOG_ROOT}')
print(f'HF_REPO={HF_REPO}')
if returncode != 0:
    raise RuntimeError(f'训练失败 exit={returncode}，日志 {run_log}')
print('status=PASS_BLINKMIMIC_T4_FORMAL')
